In [1]:

import numpy as np
from matplotlib import pyplot as plt
from utils import imgio as iio
import os
import shutil
import glob

import npy2bdv
import sys 
import re
import wx
import subprocess

import pathlib as pl


In [21]:
# Path to the folder containing the timelapse images
timelapse_path = r'g:\2PM\Amandine\240827_Doc1_00-01-00'
# Path to the folder containing the static images
static_path = r'g:\2PM\Amandine\240827_Doc1_00-00-00'


align_channel_suffix = '[6RED]*_C02'
# list files in static folder

static_files = glob.glob(static_path + f'/*{align_channel_suffix}*.tif')
static_files.sort()
print(static_files)

# list files in timelapse folder


timelapse_files = glob.glob(timelapse_path + f'/*{align_channel_suffix}*.tif')
timelapse_files.sort()
print(timelapse_files)


['g:\\2PM\\Amandine\\240827_Doc1_00-00-00\\11-32-29_Doc1_PMT - PMT [6RED] _C02_Time Time0000.ome.tif']
['g:\\2PM\\Amandine\\240827_Doc1_00-01-00\\13-17-10_Doc1_PMT - PMT [6RED] _C02_Time Time0000.ome.tif', 'g:\\2PM\\Amandine\\240827_Doc1_00-01-00\\13-17-10_Doc1_PMT - PMT [6RED] _C02_Time Time0001.ome.tif', 'g:\\2PM\\Amandine\\240827_Doc1_00-01-00\\13-17-10_Doc1_PMT - PMT [6RED] _C02_Time Time0002.ome.tif', 'g:\\2PM\\Amandine\\240827_Doc1_00-01-00\\13-17-10_Doc1_PMT - PMT [6RED] _C02_Time Time0003.ome.tif', 'g:\\2PM\\Amandine\\240827_Doc1_00-01-00\\13-17-10_Doc1_PMT - PMT [6RED] _C02_Time Time0004.ome.tif', 'g:\\2PM\\Amandine\\240827_Doc1_00-01-00\\13-17-10_Doc1_PMT - PMT [6RED] _C02_Time Time0005.ome.tif', 'g:\\2PM\\Amandine\\240827_Doc1_00-01-00\\13-17-10_Doc1_PMT - PMT [6RED] _C02_Time Time0006.ome.tif', 'g:\\2PM\\Amandine\\240827_Doc1_00-01-00\\13-17-10_Doc1_PMT - PMT [6RED] _C02_Time Time0007.ome.tif', 'g:\\2PM\\Amandine\\240827_Doc1_00-01-00\\13-17-10_Doc1_PMT - PMT [6RED] _C02_Ti

In [59]:
def make_bat(bat_dir, ds_name, idx):
    bat_file = bat_dir / f'proc_{idx}.bat'
    
    t = '@echo off\n'
    
    t += f'set dsname={ds_name}\n'
    t += f'set workdir={bat_dir}\n'
    
    # make full path to scrip as parent folder of the bat_dir
    proc_path = bat_dir.parent / 'proc.bat'
    t += f'call {proc_path}\n'
    
    with open(bat_file, 'w') as f:
        f.write(t)
        
    return bat_file

In [60]:
def make_repos_bat(bat_dir, ds_name):
    bat_file = bat_dir / f'repos.bat'
    
    t = '@echo off\n'
    
    t += f'set dsname={ds_name}\n'
    t += f'set workdir={bat_dir}\n'
    t += f'set posfile={bat_dir / "scanPos.txt"}\n'
    
    # make full path to scrip as parent folder of the bat_dir
    proc_path = bat_dir.parent / 'repos.bat'
    t += f'call {proc_path}\n'
    
    with open(bat_file, 'w') as f:
        f.write(t)
        
    return bat_file

In [84]:
assert len(static_files) == 1, 'There should be only one static image'

static_file = static_files[0]

# make tmp dir
tmp_path = pl.Path(timelapse_files[0]).parent.parent / 'tmp'
if tmp_path.exists():
    shutil.rmtree(tmp_path)
    
tmp_path.mkdir()

# make name templates:

static_file_frame = r'00-00-00_Doc1_PMT - PMT [6RED] _C02_Time Time0000.ome.tif'
timela_file_frame = r'00-00-00_Doc1_PMT - PMT [6RED] _C02_Time Time0001.ome.tif'

coords = ''
for idx, timelapse_file in enumerate(timelapse_files):
    # make dir for each timepoint
    
    ds_name = f'{idx:06d}_Doc1_00-00-00'
    timepoint_path = tmp_path / ds_name
    timepoint_path.mkdir()
    
    # copy static file
    shutil.copy(static_file, timepoint_path / static_file_frame)
    
    # copy timelapse file
    shutil.copy(timelapse_file, timepoint_path / timela_file_frame)
    
    # make bat file 
    bat_file = make_bat(tmp_path, ds_name, idx)
    
    
    # run bat file and wait for it to finish
    p = subprocess.Popen(str(bat_file), shell=True)
    p.wait()
    
    # read the file with the coordinates at the timepoint_path+'_A' / 'algn' / 'tile_000' / 'scanPos.txt'
    pos_file = tmp_path / f'{ds_name}_A' / 'algn' / 'tile_000' / 'scanPos.txt'
    with open(pos_file, 'rt') as f:
        lines = f.readlines()
    
    coords += lines[1]
    
    
# add the coordinates of the sttaic file (lines[0]) to the end of the coords
coords += lines[0]

# save the coords to a file in the tmp_path
coords_file = tmp_path / 'scanPos.txt'
with open(coords_file, 'wt') as f:
    f.write(coords)
    
    
# make joined dataset 000000_Doc1_00-00-01
ds_name_all = '000000_Doc1_00-00-01'
timela_file_frame_all = r'00-00-01_Doc1_PMT - PMT [6RED] _C02_Time Time%04d.ome.tif'



# go through all the timepoints and move timelapse files to the new folder
all_timepoint_path = tmp_path / ds_name_all
all_timepoint_path.mkdir()

for idx, timelapse_file in enumerate(timelapse_files):
    ds_name = f'{idx:06d}_Doc1_00-00-00'
    timepoint_path = tmp_path / ds_name
    
    shutil.move(timepoint_path / timela_file_frame, str(all_timepoint_path / timela_file_frame_all) % idx)

last_idx = len(timelapse_files)
    
# move static file to the new folder as well, the last one
shutil.move(timepoint_path / static_file_frame, str(all_timepoint_path / timela_file_frame_all) % (last_idx))

# create reposotitioning bat file
repos_bat = make_repos_bat(tmp_path, ds_name_all)

# run the repos.bat file
p = subprocess.Popen(str(repos_bat), shell=True)
p.wait()

# in the end one might want to move the sttatic timeframe to replicate the timelapse one at each timepoint as an addiitonal channel
out_path = all_timepoint_path.parent /(all_timepoint_path.stem+'_A')
all_tf = True

if all_tf:
    static_tf_last = out_path / (timela_file_frame_all % (last_idx))
    static_file_frame_all = timela_file_frame_all.replace(' _C0', ' _C1')
    for idx, timelapse_file in enumerate(timelapse_files):
        # copy static file (out_path / timela_file_frame_all) % (last_idx) to each timepoint
        shutil.copy(static_tf_last, out_path / (static_file_frame_all % idx))
        
    # remove the last file  -static_tf_last
    os.remove(static_tf_last)
        
# move the whole out_path folder to the final destination
timelapse_path_pl = pl.Path(timelapse_path)
final_path = timelapse_path_pl.parent / (timelapse_path_pl.stem + '_aligned')

if final_path.exists():
    print(f'{final_path} already exists, aborting'
          f'Please remove it first and rerun the script'
          f'You can find the aligned data in {out_path}')
else:
    shutil.move(out_path, final_path)
    print(f'Aligned data saved to {final_path}')
    
    #remove the tmp folder
    shutil.rmtree(tmp_path)

Aligned data saved to g:\2PM\Amandine\240827_Doc1_00-01-00_aligned
